# 🧠 Colab E — TensorFlow 3-Layer Neural Network: All 4 Variants
## Nonlinear Regression with `y = sin(x1)*cos(x2) + x3²`

| Variant | Approach |
|---|---|
| **E-i** | TF from scratch — raw variables + `GradientTape` |
| **E-ii** | TF with built-in `tf.keras.layers.Dense` |
| **E-iii** | TF Functional API (`tf.keras.Model` via inputs/outputs) |
| **E-iv** | TF High-Level API (`model.compile` + `model.fit`) |


In [ ]:
# ── Shared Setup ─────────────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)
print('TF version:', tf.__version__)

# Generate data once
N = 1000
X1 = np.random.uniform(-np.pi, np.pi, N)
X2 = np.random.uniform(-np.pi, np.pi, N)
X3 = np.random.uniform(-2, 2, N)
Y  = np.sin(X1)*np.cos(X2) + X3**2 + np.random.randn(N)*0.1
X  = np.column_stack([X1, X2, X3])

sx = StandardScaler(); sy = StandardScaler()
X_norm = sx.fit_transform(X).astype(np.float32)
Y_norm = sy.fit_transform(Y.reshape(-1,1)).astype(np.float32)
print(f'X: {X_norm.shape}, Y: {Y_norm.shape}')

---
## E-i: TensorFlow From Scratch (GradientTape + raw Variables)
No `keras.layers` — purely `tf.Variable` and `tf.GradientTape`

In [ ]:
# ── E-i SECTION 1: Raw Weight Variables ─────────────────────────────────────
def he_var(shape):
    fan_in = shape[0]
    std = np.sqrt(2.0 / fan_in)
    return tf.Variable(tf.random.normal(shape, stddev=std), trainable=True)

# 3 → 16 → 8 → 1
W1_raw = he_var([3,  16]); b1_raw = tf.Variable(tf.zeros([1,16]))
W2_raw = he_var([16,  8]); b2_raw = tf.Variable(tf.zeros([1, 8]))
W3_raw = he_var([8,   1]); b3_raw = tf.Variable(tf.zeros([1, 1]))
raw_params = [W1_raw,b1_raw, W2_raw,b2_raw, W3_raw,b3_raw]
print('Raw TF variables created')

In [ ]:
# ── E-i SECTION 2: Forward + Training with GradientTape ─────────────────────
def swish_tf(z):
    return z * tf.sigmoid(z)

def forward_raw(X):
    """Manual 3-layer forward pass using tf.einsum."""
    A1 = swish_tf(tf.einsum('ij,jk->ik', X, W1_raw) + b1_raw)
    A2 = swish_tf(tf.einsum('ij,jk->ik', A1, W2_raw) + b2_raw)
    A3 = tf.einsum('ij,jk->ik', A2, W3_raw) + b3_raw
    return A3

optimizer_raw = tf.optimizers.Adam(learning_rate=0.01)
EPOCHS = 2000
loss_hist_i = []
X_tf = tf.constant(X_norm)
Y_tf = tf.constant(Y_norm)

for epoch in range(EPOCHS):
    with tf.GradientTape() as tape:
        y_pred = forward_raw(X_tf)
        loss   = tf.reduce_mean((y_pred - Y_tf)**2)
    grads = tape.gradient(loss, raw_params)
    optimizer_raw.apply_gradients(zip(grads, raw_params))
    loss_hist_i.append(loss.numpy())
    if epoch % 400 == 0:
        print(f'[E-i] Epoch {epoch:4d} | Loss: {loss.numpy():.6f}')

plt.figure(figsize=(10,3))
plt.plot(loss_hist_i, color='crimson')
plt.title('E-i: TF Scratch (GradientTape)'); plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.grid(True); plt.show()

---
## E-ii: TensorFlow with Built-in Dense Layers (Sequential)

In [ ]:
# ── E-ii: Sequential with Dense Layers ──────────────────────────────────────
class SwishLayer(tf.keras.layers.Layer):
    """Custom Swish as a Keras Layer."""
    def call(self, x): return x * tf.sigmoid(x)

model_seq = tf.keras.Sequential([
    tf.keras.layers.Dense(16, input_shape=(3,), kernel_initializer='he_normal'),
    SwishLayer(),
    tf.keras.layers.Dense(8, kernel_initializer='he_normal'),
    SwishLayer(),
    tf.keras.layers.Dense(1)
], name='sequential_3layer')

model_seq.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
model_seq.summary()

history_ii = model_seq.fit(X_norm, Y_norm, epochs=500, batch_size=64,
                           validation_split=0.2, verbose=0)

plt.figure(figsize=(10,3))
plt.plot(history_ii.history['loss'], label='Train')
plt.plot(history_ii.history['val_loss'], label='Val')
plt.title('E-ii: TF Sequential with Dense Layers')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(True); plt.show()

---
## E-iii: TensorFlow Functional API

In [ ]:
# ── E-iii: Functional API ────────────────────────────────────────────────────
# Functional API: define graph of layers explicitly
inputs = tf.keras.Input(shape=(3,), name='inputs')
x = tf.keras.layers.Dense(16, kernel_initializer='he_normal', name='hidden1')(inputs)
x = SwishLayer(name='swish1')(x)
x = tf.keras.layers.Dense(8, kernel_initializer='he_normal', name='hidden2')(x)
x = SwishLayer(name='swish2')(x)
outputs = tf.keras.layers.Dense(1, name='output')(x)

model_func = tf.keras.Model(inputs=inputs, outputs=outputs, name='functional_3layer')
model_func.compile(optimizer=tf.keras.optimizers.Adam(0.01), loss='mse')
model_func.summary()

history_iii = model_func.fit(X_norm, Y_norm, epochs=500, batch_size=64,
                              validation_split=0.2, verbose=0)

plt.figure(figsize=(10,3))
plt.plot(history_iii.history['loss'], label='Train')
plt.plot(history_iii.history['val_loss'], label='Val')
plt.title('E-iii: TF Functional API'); plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(True); plt.show()

---
## E-iv: TensorFlow High-Level API (Model subclassing + compile/fit)

In [ ]:
# ── E-iv: Subclassed Model (High-Level API) ──────────────────────────────────
class ThreeLayerTF(tf.keras.Model):
    """
    Model-subclass style — most flexible TF API.
    Overrides build() and call() for 3-layer net.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.fc1   = tf.keras.layers.Dense(16, kernel_initializer='he_normal')
        self.fc2   = tf.keras.layers.Dense(8,  kernel_initializer='he_normal')
        self.fc3   = tf.keras.layers.Dense(1)
        self.swish = SwishLayer()

    def call(self, x, training=False):
        x = self.swish(self.fc1(x))
        x = self.swish(self.fc2(x))
        return self.fc3(x)

model_sub = ThreeLayerTF(name='subclassed_3layer')
model_sub.compile(
    optimizer=tf.keras.optimizers.Adam(0.01),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=25)
]

history_iv = model_sub.fit(X_norm, Y_norm, epochs=1000, batch_size=64,
                            validation_split=0.2, callbacks=callbacks, verbose=0)

plt.figure(figsize=(10,3))
plt.plot(history_iv.history['loss'], label='Train')
plt.plot(history_iv.history['val_loss'], label='Val')
plt.title('E-iv: TF Subclassed Model (High-Level API)')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(True); plt.show()
print(f'Best val MSE: {min(history_iv.history["val_loss"]):.6f}')

In [ ]:
# ── Comparison: All Variants Final Loss ──────────────────────────────────────
print('=== FINAL COMPARISON ===')
print(f'E-i  (TF Scratch/GradientTape):  {loss_hist_i[-1]:.6f}')
print(f'E-ii (TF Sequential Dense):      {history_ii.history["val_loss"][-1]:.6f}')
print(f'E-iii (TF Functional API):       {history_iii.history["val_loss"][-1]:.6f}')
print(f'E-iv (TF Subclassed High-Level): {min(history_iv.history["val_loss"]):.6f}')